In [1]:
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = True

import numpy as np
import xarray as xr
import cupy as cp
import numba.cuda
import cupy_xarray

SIZE = (200, 1, 987, 1920)


U = xr.DataArray(
    name="U",
    data=np.random.random(SIZE).astype(np.float32),
    dims=["time", "face", "j", "i"],
)
V = xr.DataArray(
    name="V",
    data=np.random.random(SIZE).astype(np.float32),
    dims=["time", "face", "j", "i"],
)

ds = xr.merge([U, V])
ds_gpu = ds.copy().cupy.as_cupy()

In [2]:
ds

<xarray.Dataset> Size: 3GB
Dimensions:  (time: 200, face: 1, j: 987, i: 1920)
Dimensions without coordinates: time, face, j, i
Data variables:
    U        (time, face, j, i) float32 2GB 0.7362 0.5887 ... 0.5204 0.1032
    V        (time, face, j, i) float32 2GB 0.161 0.3712 ... 0.08231 0.09366

In [13]:
%%time
u = ds["U"].data
v = ds["V"].data
uu = u * u
vv = v * v
uv = u * v
result = (uu.mean(), vv.mean(), uv.mean())

CPU times: user 1.52 s, sys: 1.19 s, total: 2.71 s
Wall time: 2.7 s


In [14]:
%%time
u = ds_gpu["U"].data
v = ds_gpu["V"].data
uu = u * u
vv = v * v
uv = u * v
result = (uu.mean(), vv.mean(), uv.mean())
cp.cuda.Stream.null.synchronize()

CPU times: user 9.22 ms, sys: 15.3 ms, total: 24.5 ms
Wall time: 22.6 ms


In [16]:
dtype = "float32"
with open("qm.cpp", "rt") as f:
    kernel_code = f.read()

U = ds_gpu["U"].data
V = ds_gpu["V"].data
    
module = cp.RawModule(code=kernel_code)
kernel = module.get_function("super_fast_kernel")

# Setup output array and call kernel
size = U.size

# Calculate grid and block dimensions for optimal occupancy
block_size = 512  # Must match the shared memory size in kernel
grid_size = min(4096, (size + block_size - 1) // block_size)

In [17]:
%%time
# Execute the kernel
results = cp.zeros(3, dtype=dtype)
kernel((grid_size,), (block_size,), (U, V, results, size))

# Compute means by dividing by size
results /= size

cp.cuda.Stream.null.synchronize()

CPU times: user 5.61 ms, sys: 0 ns, total: 5.61 ms
Wall time: 4.17 ms
